# Демо: итераторы и генераторы

Прокликай Shift+Enter каждую ячейку и посмотри, как `for` работает под капотом, что такое одноразовый итератор, чем `yield` отличается от `return`, и зачем вообще нужны ленивые вычисления. В конце — три мини-задания.

## Часть 1. Итерируемый и итератор — это разные вещи

Итерируемый объект (`list`, `tuple`, `str`, `dict`) умеет создать итератор по запросу — у него есть метод `__iter__()`. Итератор — это объект, выдающий элементы по одному через `__next__()`. Список **не** итератор: его нельзя «прокручивать вручную».

In [1]:
items = [10, 20, 30]

# items — итерируемый, но не итератор
try:
    next(items)
except TypeError as e:
    print(f"TypeError: {e}")

# Получим итератор через iter()
it = iter(items)
print(f"тип: {type(it).__name__}")
print(next(it))    # 10
print(next(it))    # 20
print(next(it))    # 30

TypeError: 'list' object is not an iterator
тип: list_iterator
10
20
30


А что если позвать `next` ещё раз? Итератор закончился — Python поднимает `StopIteration`. Цикл `for` ловит это исключение автоматически и тихо выходит из цикла.

In [2]:
try:
    next(it)
except StopIteration:
    print("итератор исчерпан")

итератор исчерпан


Под капотом цикл `for x in items:` делает три действия: получает итератор через `iter(items)`, в цикле зовёт `next(...)`, ловит `StopIteration`. Покажем, как `for` выглядит без синтаксического сахара:

In [3]:
items = [10, 20, 30]

it = iter(items)
while True:
    try:
        x = next(it)
    except StopIteration:
        break
    print(x)

10
20
30


## Часть 2. Итератор одноразовый

Итератор — это «курсор», запоминающий позицию. После прохода его нельзя перемотать. Нужно начать сначала — создавайте новый итератор через `iter()`.

In [4]:
items = [1, 2, 3]
it = iter(items)

# Первый проход — обычный
for x in it:
    print(f"первый проход: {x}")

# Второй проход — пусто! итератор уже исчерпан
for x in it:
    print(f"второй проход: {x}")
print("второй проход не напечатал ничего")

первый проход: 1
первый проход: 2
первый проход: 3
второй проход не напечатал ничего


## Часть 3. Утиная типизация — `for` работает с чем угодно

Цикл `for` не проверяет тип объекта. Он просит у объекта итератор. Если объект соблюдает протокол итератора — `for` работает. Это и есть duck typing: «если ходит как утка и крякает как утка — это утка».

In [5]:
# Все эти объекты ведут себя как итерируемые — for работает одинаково
for char in "abc":
    print(char, end=" ")
print()

for key in {"a": 1, "b": 2, "c": 3}:
    print(key, end=" ")
print()

for n in range(3):
    print(n, end=" ")
print()

a b c 
a b c 
0 1 2 


## Часть 4. Генератор — функция с `yield`

Если в теле функции встретить `yield` вместо `return`, функция превращается в генератор. Вызов `gen()` не выполняет тело — он создаёт генератор-объект (это итератор). Тело начинает выполняться только при первом `next()`.

In [6]:
def count_to(n):
    print(f"  начали считать до {n}")
    i = 0
    while i < n:
        print(f"  yield {i}")
        yield i
        i += 1
    print("  тело закончилось")

# Вызвали — но тело ещё не выполнилось
gen = count_to(3)
print(f"тип: {type(gen).__name__}")

# Каждый next двигает функцию до следующего yield
print(next(gen))    # тело начинает работать
print(next(gen))
print(next(gen))

# А что после последнего yield? StopIteration
try:
    next(gen)
except StopIteration:
    print("StopIteration — генератор исчерпан")

тип: generator
  начали считать до 3
  yield 0
0
  yield 1
1
  yield 2
2
  тело закончилось
StopIteration — генератор исчерпан


Семантика `yield`: отдать значение, **заморозить** функцию на этой строке (с локальными переменными), при следующем `next` — продолжить выполнение со следующей строки. После исчерпания — `StopIteration`.

## Часть 5. Ленивые vs жадные вычисления

Список считает все элементы сразу и держит их в памяти. Генератор считает следующий элемент только когда его попросят, и не помнит предыдущие. На больших данных это разница между «занять 10 ГБ памяти» и «занять 100 байт».

In [7]:
# Список: сразу 1 000 000 чисел в памяти
import sys

list_version = [x ** 2 for x in range(1_000_000)]
gen_version = (x ** 2 for x in range(1_000_000))

print(f"список:    {sys.getsizeof(list_version):>10} байт")
print(f"генератор: {sys.getsizeof(gen_version):>10} байт")
print(f"первые 5 из генератора: {[next(gen_version) for _ in range(5)]}")

список:       8448728 байт
генератор:        112 байт
первые 5 из генератора: [0, 1, 4, 9, 16]


Генераторное выражение `(expr for x in iterable)` — ленивый аналог list comprehension. Синтаксис тот же, только круглые скобки вместо квадратных. Передавать как аргумент `sum(x ** 2 for x in nums)` можно без скобок.

In [8]:
numbers = range(10)

# Сумма квадратов чётных — без промежуточного списка
result = sum(x ** 2 for x in numbers if x % 2 == 0)
print(result)    # 0 + 4 + 16 + 36 + 64 = 120

120


## Часть 6. `yield from` — пробросить вложенный генератор

Если внутри генератора нужно отдать все элементы другого генератора — пишем `yield from`. Эквивалент `for x in inner: yield x`, но короче.

In [9]:
def chain(*iterables):
    for it in iterables:
        yield from it    # пробрасываем все элементы каждого

result = list(chain([1, 2, 3], "abc", (10, 20)))
print(result)    # [1, 2, 3, 'a', 'b', 'c', 10, 20]

[1, 2, 3, 'a', 'b', 'c', 10, 20]


## Мини-задания

Три коротких упражнения. Подсказок к именам и методам нет — вспомни сам.

**Задание 1.** Напиши генератор `take(n, iterable)`, который выдаёт первые `n` элементов любого итерируемого объекта.

**Задание 2.** Напиши генератор `evens_up_to(limit)`, который выдаёт чётные числа `0, 2, 4, ...` до `limit` (не включая). Проверь: `list(evens_up_to(10))` должно вернуть `[0, 2, 4, 6, 8]`.

**Задание 3.** Что напечатает код ниже? Сначала угадай, потом запусти.

In [10]:
# Задание 1
# def take(n, iterable):
#     ...

# Проверка:
# print(list(take(3, [10, 20, 30, 40, 50])))  # [10, 20, 30]
# print(list(take(2, 'привет')))               # ['п', 'р']


In [11]:
# Задание 2
# def evens_up_to(limit):
#     ...

# Проверка:
# print(list(evens_up_to(10)))   # [0, 2, 4, 6, 8]
# print(list(evens_up_to(0)))    # []


In [12]:
# Задание 3 — твой прогноз для каждой строки впиши в комментарий
def gen():
    yield 1
    yield 2
    yield 3

g = gen()
print(list(g))   # ?
print(list(g))   # ?

# Подсказка: вспомни про одноразовость итератора


[1, 2, 3]
[]
